# Testes de contrato da camada Silver

Notebook de regressão com DataFrames Spark sintéticos e isolados. As asserções verificam os contratos IAM/IGA, as rejeições de qualidade e o contexto de acesso, sem depender do conteúdo mutável do warehouse.

In [1]:
import sys
from datetime import date
from pathlib import Path

from pyspark.sql import functions as F

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / "pyproject.toml").exists()), None
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook dentro do projeto sod-platform.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from sod_platform.access_intelligence.context.engine import build_access_context
from sod_platform.bronze.ingestion.spark import create_spark_session
from sod_platform.silver.preparation import prepare_silver

spark = create_spark_session(PROJECT_ROOT, "silver-contract-tests")
print(f"Spark pronto: {spark.version}; projeto: {PROJECT_ROOT}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/26 01:14:56 WARN Utils: Your hostname, Note-LAS, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/26 01:14:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/lucas/Projetos/sod-platform/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lucas/.ivy2.5.2/cache
The jars for the packages stored in: /home/lucas/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eb936482-b10a-4dd7-8810-888cb2ca8cac;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
:: resolution report :: resolve 103ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runt

Spark pronto: 4.1.3; projeto: /home/lucas/Projetos/sod-platform


In [2]:
from sod_platform.silver.contract import INPUT_SCHEMAS

records = {
    "identity_master": {
        "identidade_id": "ID-1",
        "comunidade": "Credito",
        "squad": "nucleo_alpha",
        "cargo": "analista",
        "tipo_identidade": "employee",
        "gestor": "GESTOR-1",
        "data_entrada_comunidade_atual": "2023-01-15",
        "status_identidade": "active",
        "data_desligamento": None,
    },
    "identity_directory": {
        "account_id": "ACC-1",
        "identidade_id": "ID-1",
        "status_conta": "active",
        "tipo_conta": "employee",
        "data_criacao": "2020-01-01",
        "ultima_autenticacao": "2025-01-30",
        "data_bloqueio": None,
    },
    "iga_entitlements": {
        "entitlement_id": "ENT-1",
        "sigla_id": "SIG-1",
        "comunidade_dona_sigla": "Credito",
        "birthright": "false",
        "sigla_publica": "false",
        "application_owner": "OWNER-1",
        "criticidade": "HIGH",
        "classificacao_dado": "RESTRICTED",
        "privileged": "false",
        "regulatory_scope": "BACEN",
    },
    "application_catalog": {
        "sigla_id": "SIG-1",
        "nome_sistema": "Sistema de teste",
        "owner": "OWNER-1",
        "comunidade": "Credito",
        "criticidade": "HIGH",
        "classificacao_dado": "RESTRICTED",
        "privileged": "false",
        "regulatory_scope": "BACEN",
    },
    "iga_access_assignments": {
        "grant_id": "GR-1",
        "identidade_id": "ID-1",
        "entitlement_id": "ENT-1",
        "request_id": "REQ-1",
        "data_concessao": "2024-01-02",
        "data_revogacao": None,
        "status_concessao": "active",
        "tipo_atribuicao": "direct",
        "ultimo_uso": "2025-01-20",
    },
    "iga_access_requests": {
        "request_id": "REQ-1",
        "grant_id": "GR-1",
        "identidade_id": "ID-1",
        "entitlement_id": "ENT-1",
        "solicitante": "ID-1",
        "data_solicitacao": "2024-01-01",
        "status_solicitacao": "APPROVED",
        "aprovador": "GESTOR-1",
        "data_aprovacao": "2024-01-02",
        "motivo": "Acesso necessário",
    },
    "access_certifications": {
        "campaign_id": "CAMP-1",
        "identidade_id": "ID-1",
        "entitlement_id": "ENT-1",
        "data_revisao": "2025-01-15",
        "revisor": "GESTOR-1",
        "decisao": "MAINTAIN",
        "justificativa": "Acesso confirmado",
    },
}

raw = {}
for name, record in records.items():
    schema = ", ".join(f"{column} string" for column in INPUT_SCHEMAS[name])
    frame = spark.createDataFrame([record], schema)
    raw[name] = frame.withColumn("_source_file", F.lit(f"synthetic/{name}")).withColumn(
        "_ingestion_id", F.lit("notebook-fixture")
    )

print(f"Fixtures IAM/IGA criadas: {sorted(raw)}")

Fixtures IAM/IGA criadas: ['access_certifications', 'application_catalog', 'identity_directory', 'identity_master', 'iga_access_assignments', 'iga_access_requests', 'iga_entitlements']


In [3]:
config = {
    "date_formats": ["yyyy-MM-dd"],
    "domains": {
        "tipo_identidade": ["employee", "contractor"],
        "tipo_atribuicao": ["direct", "inherited"],
    },
    "tables": {name: f"bronze.{name}" for name in raw},
}

valid, rejects, transformed = prepare_silver(raw, config, "notebook-valid")
expected_tables = set(records)
assert expected_tables.issubset(valid), (
    f"Datasets ausentes: {expected_tables - set(valid)}"
)
assert all(valid[name].count() == 1 for name in expected_tables)
assert sum(frame.count() for frame in rejects) == 0
print(f"Contrato válido: {len(expected_tables)} datasets; rejeições: 0")

26/09/26 01:15:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Contrato válido: 7 datasets; rejeições: 0


In [4]:
context = build_access_context(valid, date(2025, 2, 1))
assert context.count() == 1
row = context.first()
assert row.identidade_id == "ID-1"
assert row.entitlement_id == "ENT-1"
assert row.approval_exists is True
assert row.cross_community is False
assert row._accesses_lineage is not None
print("Contexto validado: uma linha, aprovação associada e lineage preservada")

Contexto validado: uma linha, aprovação associada e lineage preservada


In [5]:
bad_identity = dict(records["identity_master"], tipo_identidade="robot")
bad_frame = spark.createDataFrame(
    [bad_identity],
    ", ".join(f"{column} string" for column in INPUT_SCHEMAS["identity_master"]),
).withColumn("_source_file", F.lit("synthetic/identity_master-bad"))
bad_raw = {**raw, "identity_master": bad_frame}
_, bad_rejects, _ = prepare_silver(bad_raw, config, "notebook-invalid-domain")

rejection_codes = {
    row.error_code
    for rejected in bad_rejects
    for row in rejected.select("error_code").distinct().collect()
}
assert "INVALID_DOMAIN" in rejection_codes, rejection_codes
print(f"Quarentena validada: {sorted(rejection_codes)}")

Quarentena validada: ['INVALID_DOMAIN', 'INVALID_IDENTITY_REFERENCE']


## Resultado esperado

O notebook está aprovado quando todas as células de código terminam sem exceção: datasets IAM/IGA conformes permanecem válidos, o contexto contém um registro único e valores fora do domínio são enviados à quarentena.